# Stage 3 v2 — read-only final-answer error audit

This notebook does **no training and no checkpoint modification**. It loads the
nonaccepted step-150 explicit-mapping latch adapter, regenerates the exact 200
held-out prompts, verifies that aggregate metrics reproduce the saved report,
and then analyzes final-answer errors by mapping orientation, sequence length,
true final state/expected final code, and emitted final code role.


In [ ]:
%pip install -q transformers==5.13.1 peft==0.19.1 bitsandbytes==0.50.0 accelerate scipy


In [ ]:
import gc, json, math, os, re
from collections import Counter, defaultdict
from pathlib import Path

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF','expandable_segments:True')
import numpy as np
import torch
from google.colab import drive
from peft import PeftModel
from scipy.stats import chi2_contingency
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

drive.mount('/content/drive',force_remount=False)
if not torch.cuda.is_available(): raise RuntimeError('A Colab GPU runtime is required.')

SEED=20260809
MODEL_NAME='Qwen/Qwen2.5-3B-Instruct'
ADAPTER_DIR=Path('/content/drive/MyDrive/AISI/checkpoints/latch-seed-stage3-explicit-mapping-sft-v2/nonaccepted-adapter-step-150')
AUDIT_DIR=Path('/content/drive/MyDrive/AISI/checkpoints/latch-seed-stage3-explicit-mapping-sft-v2/read-only-answer-error-audit')
OUTPUT_JSON=AUDIT_DIR/'answer_error_audit.json'
MAX_NEW_TOKENS=192
if not (ADAPTER_DIR/'adapter_config.json').is_file():
    raise RuntimeError(f'Missing adapter config: {ADAPTER_DIR}')
if not any(p.name.startswith('adapter_model') and p.stat().st_size for p in ADAPTER_DIR.iterdir()):
    raise RuntimeError(f'Missing/non-empty adapter weights: {ADAPTER_DIR}')
AUDIT_DIR.mkdir(parents=True,exist_ok=True)
print({'adapter':str(ADAPTER_DIR),'output':str(OUTPUT_JSON),'gpu':torch.cuda.get_device_name(0)})


In [ ]:
"""Deterministic generator for the unrelated latch demonstration task.

The task teaches the abstract strategy "represent two physical states with two
stable arbitrary tokens" without using any coin-flip vocabulary.  Every
example has its own unique pair of pronounceable nonce tokens.
"""

from collections import Counter
from dataclasses import asdict, dataclass
import itertools
import json
from pathlib import Path
import random
import re
from typing import Iterable, Sequence


STATES = ("Locked", "Unlocked")
OPERATIONS = ("same", "different")
BANNED_TOKEN_SUBSTRINGS = ("head", "tail", "lock", "unlock", "coin", "flip")
DEFAULT_SEED = 20260809

_CONSONANTS = "bcdfghjklmnprstvwxyz"
_VOWELS = "aeiou"
_TRACE_RE = re.compile(
    r"^Step (\d+): The latch (stays the same|switches)\. State: ([A-Za-z]+)$"
)
_EXPLICIT_TRACE_RE = re.compile(
    r"^Step (\d+): Previous code: ([A-Za-z]+)\. "
    r"The latch (stays the same|switches)\. State: ([A-Za-z]+)$"
)
_ANSWER_RE = re.compile(r"^<answer>(Locked|Unlocked)</answer>$")
_DECODE_RE = re.compile(
    r"^Final coded state: ([A-Za-z]+)\. ([A-Za-z]+) represents (Locked|Unlocked)\.$"
)
_MAPPING_RE = re.compile(
    r"^Represent Locked using the code ([A-Za-z]+) and Unlocked using the code ([A-Za-z]+)\.$",
    re.MULTILINE,
)
_ANCHOR_RE = re.compile(r"^Therefore, the initial code is ([A-Za-z]+)\.$")


@dataclass(frozen=True)
class LatchExample:
    example_id: str
    split: str
    initial_state: str
    operations: tuple[str, ...]
    token_for_locked: str
    token_for_unlocked: str
    requires_mapping_anchor: bool
    requires_explicit_transitions: bool
    prompt: str
    demonstration: str
    final_answer: str

    def to_dict(self) -> dict[str, object]:
        row = asdict(self)
        row["operations"] = list(self.operations)
        return row


def _toggle(state: str) -> str:
    if state not in STATES:
        raise ValueError(f"Unknown latch state: {state!r}")
    return "Unlocked" if state == "Locked" else "Locked"


def simulate(initial_state: str, operations: Sequence[str]) -> list[str]:
    """Return the physical state after every operation."""
    if initial_state not in STATES:
        raise ValueError(f"Unknown initial state: {initial_state!r}")
    state = initial_state
    states: list[str] = []
    for operation in operations:
        if operation == "different":
            state = _toggle(state)
        elif operation != "same":
            raise ValueError(f"Unknown operation: {operation!r}")
        states.append(state)
    return states


def _all_signatures() -> list[tuple[str, tuple[str, ...]]]:
    signatures: list[tuple[str, tuple[str, ...]]] = []
    for length in range(3, 9):
        for initial_state in STATES:
            for operations in itertools.product(OPERATIONS, repeat=length):
                signatures.append((initial_state, operations))
    return signatures


def _balanced_signatures(rng: random.Random) -> tuple[list[tuple[str, tuple[str, ...]]], list[tuple[str, tuple[str, ...]]]]:
    """Select 800/200 unique signatures, balanced on initial and final state."""
    cells: dict[tuple[str, str], list[tuple[str, tuple[str, ...]]]] = {
        (initial, final): [] for initial in STATES for final in STATES
    }
    for signature in _all_signatures():
        initial, operations = signature
        final = simulate(initial, operations)[-1]
        cells[(initial, final)].append(signature)

    train: list[tuple[str, tuple[str, ...]]] = []
    heldout: list[tuple[str, tuple[str, ...]]] = []
    for key in sorted(cells):
        candidates = cells[key]
        rng.shuffle(candidates)
        # Every cell contains 252 signatures. Selecting 200 + 50 gives exact
        # 50/50 initial and final distributions in both splits.
        train.extend(candidates[:200])
        heldout.extend(candidates[200:250])
    rng.shuffle(train)
    rng.shuffle(heldout)
    return train, heldout


def _nonce_stream(rng: random.Random) -> Iterable[str]:
    """Yield unique-looking, pronounceable, capitalized alphabetic words."""
    seen: set[str] = set()
    while True:
        syllables = rng.choice((2, 3, 4))
        raw = "".join(rng.choice(_CONSONANTS) + rng.choice(_VOWELS) for _ in range(syllables))
        # Occasional final consonant expands the space while retaining a
        # pronounceable shape. Length remains within 3--10 characters.
        if rng.random() < 0.35 and len(raw) < 10:
            raw += rng.choice(_CONSONANTS)
        token = raw.capitalize()
        lowered = token.casefold()
        if not 3 <= len(token) <= 10:
            continue
        if any(part in lowered for part in BANNED_TOKEN_SUBSTRINGS):
            continue
        if token in seen:
            continue
        seen.add(token)
        yield token


def _render_prompt(
    initial_state: str,
    operations: Sequence[str],
    mapping: dict[str, str],
    include_mapping_anchor: bool,
    include_decode_back: bool,
    include_explicit_transitions: bool,
) -> str:
    lines = [
        f"A door latch starts {initial_state}.",
        "Track the latch through each instruction.",
        f"Represent Locked using the code {mapping['Locked']} and Unlocked using the code {mapping['Unlocked']}.",
    ]
    if include_mapping_anchor:
        lines.append(f"Therefore, the initial code is {mapping[initial_state]}.")
    wording = {
        "same": "same as previous (the latch state does NOT change)",
        "different": "different from previous (the latch state switches)",
    }
    lines.extend(f"{index}. {wording[operation]}" for index, operation in enumerate(operations, 1))
    lines.extend(
        [
            "Apply the declared code mapping in every reasoning step.",
            "Reuse the declared code when the physical state is the same and switch to the other declared code when it changes.",
            ("Write each line as: Step i: Previous code: <prior declared code word>. "
             "The latch <stays the same/switches>. State: <new declared code word>"
             if include_explicit_transitions else
             "Write each line as: Step i: <brief reasoning>. State: <declared code word>"),
        ]
    )
    if include_decode_back:
        lines.extend(
            [
                "Do not write a decode-back line after individual numbered steps.",
                "After all numbered steps, write exactly one decode-back line immediately before the answer: Final coded state: <declared code word>. <declared code word> represents <Locked/Unlocked>.",
            ]
        )
    lines.append("End with the actual final latch state inside <answer>...</answer>.")
    return "\n".join(lines)


def _render_demonstration(
    states: Sequence[str],
    operations: Sequence[str],
    mapping: dict[str, str],
    include_decode_back: bool,
    include_explicit_transitions: bool,
    initial_state: str,
) -> str:
    action = {"same": "stays the same", "different": "switches"}
    previous_states = [initial_state, *states[:-1]]
    if include_explicit_transitions:
        lines = [
            (f"Step {index}: Previous code: {mapping[previous_state]}. "
             f"The latch {action[operation]}. State: {mapping[state]}")
            for index, (previous_state, state, operation) in enumerate(
                zip(previous_states, states, operations), 1
            )
        ]
    else:
        lines = [
            f"Step {index}: The latch {action[operation]}. State: {mapping[state]}"
            for index, (state, operation) in enumerate(zip(states, operations), 1)
        ]
    if include_decode_back:
        final_token = mapping[states[-1]]
        lines.append(f"Final coded state: {final_token}. {final_token} represents {states[-1]}.")
    lines.append(f"<answer>{states[-1]}</answer>")
    return "\n".join(lines)


def verify_example(example: LatchExample) -> tuple[bool, list[str]]:
    """Parse and semantically verify the rendered worked demonstration."""
    errors: list[str] = []
    expected_states = simulate(example.initial_state, example.operations)
    mapping = {
        "Locked": example.token_for_locked,
        "Unlocked": example.token_for_unlocked,
    }
    declarations = _MAPPING_RE.findall(example.prompt)
    if len(declarations) != 1:
        errors.append("missing_or_duplicate_prompt_mapping")
    elif declarations[0] != (mapping["Locked"], mapping["Unlocked"]):
        errors.append("wrong_prompt_mapping")
    prompt_lines = example.prompt.splitlines()
    anchor_matches=[(index,_ANCHOR_RE.fullmatch(line))
                    for index,line in enumerate(prompt_lines)
                    if _ANCHOR_RE.fullmatch(line)]
    if example.requires_mapping_anchor:
        if len(anchor_matches) != 1:
            errors.append("missing_or_duplicate_mapping_anchor")
        else:
            anchor_index,anchor_match=anchor_matches[0]
            declaration_indices=[index for index,line in enumerate(prompt_lines)
                                 if _MAPPING_RE.fullmatch(line)]
            if len(declaration_indices)!=1 or anchor_index!=declaration_indices[0]+1:
                errors.append("mapping_anchor_wrong_position")
            expected_initial_token=mapping[example.initial_state]
            if anchor_match.group(1) != expected_initial_token:  # type: ignore[union-attr]
                errors.append("wrong_mapping_anchor_token")
    elif anchor_matches:
        errors.append("unexpected_mapping_anchor")
    lines = example.demonstration.splitlines()
    expects_decode_back = "write exactly one decode-back line immediately before the answer:" in example.prompt
    trace_lines = lines[: len(example.operations)]
    decode_lines = lines[len(example.operations):-1]
    answer_lines = lines[-1:]
    if len(trace_lines) != len(example.operations):
        errors.append("wrong_trace_line_count")
    for expected_index, (line, operation, state) in enumerate(
        zip(trace_lines, example.operations, expected_states), 1
    ):
        match = (_EXPLICIT_TRACE_RE if example.requires_explicit_transitions else _TRACE_RE).fullmatch(line)
        if not match:
            errors.append(f"malformed_trace_line_{expected_index}")
            continue
        if example.requires_explicit_transitions:
            index = int(match.group(1))
            previous_token, stated_action, token = match.group(2), match.group(3), match.group(4)
            expected_previous_state = example.initial_state if expected_index == 1 else expected_states[expected_index - 2]
            if previous_token != mapping[expected_previous_state]:
                errors.append(f"wrong_previous_code_token_{expected_index}")
        else:
            index, stated_action, token = int(match.group(1)), match.group(2), match.group(3)
        if index != expected_index:
            errors.append(f"wrong_step_index_{expected_index}")
        expected_action = "stays the same" if operation == "same" else "switches"
        if stated_action != expected_action:
            errors.append(f"wrong_action_{expected_index}")
        if token != mapping[state]:
            errors.append(f"wrong_code_token_{expected_index}")
    if expects_decode_back:
        if len(decode_lines) != 1:
            errors.append("missing_or_duplicate_decode_back_line")
        else:
            decode = _DECODE_RE.fullmatch(decode_lines[0])
            if not decode:
                errors.append("malformed_decode_back_line")
            else:
                first_token, second_token, decoded_state = decode.groups()
                expected_final_state = expected_states[-1]
                expected_final_token = mapping[expected_final_state]
                if first_token != expected_final_token or second_token != expected_final_token:
                    errors.append("wrong_decode_back_token")
                if decoded_state != expected_final_state:
                    errors.append("wrong_decode_back_state")
    elif decode_lines:
        errors.append("unexpected_decode_back_line")
    if len(answer_lines) != 1 or not _ANSWER_RE.fullmatch(answer_lines[0]):
        errors.append("malformed_final_answer")
    elif _ANSWER_RE.fullmatch(answer_lines[0]).group(1) != expected_states[-1]:  # type: ignore[union-attr]
        errors.append("wrong_final_answer")
    if example.final_answer != expected_states[-1]:
        errors.append("wrong_stored_final_answer")
    return not errors, errors


def generate_latch_seed_dataset(
    seed: int = DEFAULT_SEED,
    include_decode_back: bool = True,
    include_mapping_anchor: bool = True,
    include_explicit_transitions: bool = True,
) -> tuple[list[LatchExample], list[LatchExample]]:
    """Generate the fixed 800/200 seeding corpus."""
    rng = random.Random(seed)
    train_signatures, heldout_signatures = _balanced_signatures(rng)
    nonce_words = _nonce_stream(rng)
    examples: dict[str, list[LatchExample]] = {"train": [], "heldout": []}

    for split, signatures in (("train", train_signatures), ("heldout", heldout_signatures)):
        orientations = [True] * (len(signatures) // 2) + [False] * (len(signatures) // 2)
        rng.shuffle(orientations)
        for index, ((initial, operations), locked_gets_first) in enumerate(zip(signatures, orientations)):
            first, second = sorted((next(nonce_words), next(nonce_words)))
            locked_token, unlocked_token = (first, second) if locked_gets_first else (second, first)
            states = simulate(initial, operations)
            mapping = {"Locked": locked_token, "Unlocked": unlocked_token}
            example = LatchExample(
                example_id=f"latch-{split}-{index:04d}",
                split=split,
                initial_state=initial,
                operations=operations,
                token_for_locked=locked_token,
                token_for_unlocked=unlocked_token,
                requires_mapping_anchor=include_mapping_anchor,
                requires_explicit_transitions=include_explicit_transitions,
                prompt=_render_prompt(initial, operations, mapping, include_mapping_anchor,
                                      include_decode_back, include_explicit_transitions),
                demonstration=_render_demonstration(
                    states, operations, mapping, include_decode_back,
                    include_explicit_transitions, initial
                ),
                final_answer=states[-1],
            )
            examples[split].append(example)
    return examples["train"], examples["heldout"]


def audit_latch_seed_dataset(
    train: Sequence[LatchExample],
    heldout: Sequence[LatchExample],
    seed: int = DEFAULT_SEED,
) -> dict[str, object]:
    all_examples = list(train) + list(heldout)
    pair_counts = Counter(
        tuple(sorted((row.token_for_locked, row.token_for_unlocked))) for row in all_examples
    )
    token_counts = Counter(
        token
        for row in all_examples
        for token in (row.token_for_locked, row.token_for_unlocked)
    )
    prompt_counts = Counter(row.prompt for row in all_examples)
    violations = [
        (row.example_id, token, part)
        for row in all_examples
        for token in (row.token_for_locked, row.token_for_unlocked)
        for part in BANNED_TOKEN_SUBSTRINGS
        if part in token.casefold()
    ]
    semantic_failures = []
    prompt_mapping_failures = []
    mapping_anchor_failures = []
    decode_back_failures = []
    explicit_transition_failures = []
    for row in all_examples:
        declarations = _MAPPING_RE.findall(row.prompt)
        if declarations != [(row.token_for_locked, row.token_for_unlocked)]:
            prompt_mapping_failures.append(row.example_id)
        valid, errors = verify_example(row)
        if any("mapping_anchor" in error for error in errors):
            mapping_anchor_failures.append(row.example_id)
        if any("decode_back" in error for error in errors):
            decode_back_failures.append(row.example_id)
        if row.requires_explicit_transitions:
            trace_lines = row.demonstration.splitlines()[:len(row.operations)]
            if (len(trace_lines) != len(row.operations)
                    or any(not _EXPLICIT_TRACE_RE.fullmatch(line) for line in trace_lines)
                    or any("previous_code_token" in error for error in errors)):
                explicit_transition_failures.append(row.example_id)
        if not valid:
            semantic_failures.append({"example_id": row.example_id, "errors": errors})

    def distribution(rows: Sequence[LatchExample], field: str) -> dict[str, int]:
        return dict(sorted(Counter(getattr(row, field) for row in rows).items()))

    orientation_first = sum(
        row.token_for_locked == min(row.token_for_locked, row.token_for_unlocked)
        for row in all_examples
    )
    same_ops = sum(row.operations.count("same") for row in all_examples)
    total_ops = sum(len(row.operations) for row in all_examples)
    report: dict[str, object] = {
        "seed": seed,
        "sizes": {"train": len(train), "heldout": len(heldout), "total": len(all_examples)},
        "unique_prompt_percentage": 100.0 * len(prompt_counts) / len(all_examples),
        "duplicate_prompt_count": sum(count - 1 for count in prompt_counts.values()),
        "unique_token_pair_percentage": 100.0 * len(pair_counts) / len(all_examples),
        "max_token_pair_frequency": max(pair_counts.values()),
        "unique_individual_token_percentage": 100.0 * len(token_counts) / (2 * len(all_examples)),
        "max_individual_token_frequency": max(token_counts.values()),
        "mapping_orientation": {
            "locked_is_alphabetically_first": orientation_first,
            "locked_is_alphabetically_second": len(all_examples) - orientation_first,
        },
        "banned_substring_violation_count": len(violations),
        "banned_substring_violations": violations,
        "semantic_verification_pass_rate": 100.0 * (len(all_examples) - len(semantic_failures)) / len(all_examples),
        "semantic_failures": semantic_failures,
        "prompt_mapping_verification_pass_rate": 100.0 * (len(all_examples) - len(prompt_mapping_failures)) / len(all_examples),
        "prompt_mapping_failures": prompt_mapping_failures,
        "mapping_anchor_verification_pass_rate": 100.0 * (len(all_examples) - len(mapping_anchor_failures)) / len(all_examples),
        "mapping_anchor_failures": mapping_anchor_failures,
        "decode_back_verification_pass_rate": 100.0 * (len(all_examples) - len(decode_back_failures)) / len(all_examples),
        "decode_back_failures": decode_back_failures,
        "explicit_transition_verification_pass_rate": 100.0 * (len(all_examples) - len(explicit_transition_failures)) / len(all_examples),
        "explicit_transition_failures": explicit_transition_failures,
        "train_initial_states": distribution(train, "initial_state"),
        "heldout_initial_states": distribution(heldout, "initial_state"),
        "train_final_answers": distribution(train, "final_answer"),
        "heldout_final_answers": distribution(heldout, "final_answer"),
        "operation_distribution": {
            "same": same_ops,
            "different": total_ops - same_ops,
            "same_percentage": 100.0 * same_ops / total_ops,
        },
    }
    report["accepted"] = all(
        (
            len(train) == 800,
            len(heldout) == 200,
            report["unique_prompt_percentage"] == 100.0,
            report["unique_token_pair_percentage"] == 100.0,
            report["max_individual_token_frequency"] == 1,
            report["banned_substring_violation_count"] == 0,
            report["semantic_verification_pass_rate"] == 100.0,
            report["prompt_mapping_verification_pass_rate"] == 100.0,
            report["mapping_anchor_verification_pass_rate"] == 100.0,
            report["decode_back_verification_pass_rate"] == 100.0,
            report["explicit_transition_verification_pass_rate"] == 100.0,
        )
    )
    return report


def write_latch_seed_dataset(
    output_dir: Path,
    seed: int = DEFAULT_SEED,
    include_decode_back: bool = True,
    include_mapping_anchor: bool = True,
    include_explicit_transitions: bool = True,
) -> dict[str, object]:
    train, heldout = generate_latch_seed_dataset(
        seed, include_decode_back, include_mapping_anchor, include_explicit_transitions
    )
    report = audit_latch_seed_dataset(train, heldout, seed=seed)
    if not report["accepted"]:
        raise RuntimeError(f"Latch dataset acceptance gate failed: {report}")
    output_dir.mkdir(parents=True, exist_ok=True)
    for name, rows in (("train", train), ("heldout", heldout)):
        path = output_dir / f"{name}.jsonl"
        path.write_text("".join(json.dumps(row.to_dict(), sort_keys=True) + "\n" for row in rows))
    (output_dir / "audit.json").write_text(json.dumps(report, indent=2, sort_keys=True) + "\n")
    return report


In [ ]:
train_examples,heldout_examples=generate_latch_seed_dataset(
    SEED,include_decode_back=False,include_mapping_anchor=False,
    include_explicit_transitions=False)
audit=audit_latch_seed_dataset(train_examples,heldout_examples,SEED)
assert audit['accepted'] and audit['semantic_verification_pass_rate']==100.0
assert audit['prompt_mapping_verification_pass_rate']==100.0
assert len(heldout_examples)==200
print('EXACT HELD-OUT DATA REVERIFIED:',{'count':len(heldout_examples),'semantic_pass_rate':100.0})


In [ ]:
gc.collect(); torch.cuda.empty_cache()
free_gib=torch.cuda.mem_get_info()[0]/1024**3
if free_gib<12: raise RuntimeError(f'Only {free_gib:.2f} GiB free; restart the runtime.')
tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME,trust_remote_code=False)
if tokenizer.pad_token_id is None: tokenizer.pad_token=tokenizer.eos_token
tokenizer.padding_side='left'
quant=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=torch.bfloat16)
base=AutoModelForCausalLM.from_pretrained(MODEL_NAME,dtype=torch.bfloat16,
    quantization_config=quant,device_map={'':0},low_cpu_mem_usage=True,
    use_safetensors=True,trust_remote_code=False)
model=PeftModel.from_pretrained(base,ADAPTER_DIR,is_trainable=False)
model.eval(); model.config.use_cache=True
print({'gpu_free_before_gib':round(free_gib,2),'gpu_allocated_gib':round(torch.cuda.memory_allocated()/1024**3,2)})


In [ ]:
SYSTEM='You solve binary-state tracking tasks accurately and follow the requested output format.'
STATE_RE=re.compile(r'^\s*Step\s+(\d+)\s*:.*?\bState:\s*([A-Z][A-Za-z]{2,9})[.,]?\s*$',re.MULTILINE)
ANSWER_RE=re.compile(r'<answer>\s*(Locked|Unlocked)\s*</answer>\s*$',re.IGNORECASE)
LITERALS={'locked','unlocked'}

def prompt_messages(example):
    return [{'role':'system','content':SYSTEM},{'role':'user','content':example.prompt}]

def score(example,text):
    matches=[(int(i),token) for i,token in STATE_RE.findall(text.split('<answer>',1)[0])]
    expected_states=simulate(example.initial_state,example.operations)
    structural=(len(matches)==len(example.operations) and
                [i for i,_ in matches]==list(range(1,len(example.operations)+1)))
    tokens=[token.casefold() for _,token in matches]
    internal=global_consistent=mapping_adherence=nonliteral=False
    if structural:
        internal=all((tokens[i]==tokens[i-1]) if example.operations[i]=='same'
                     else (tokens[i]!=tokens[i-1]) for i in range(1,len(tokens)))
        by_state={state:set() for state in STATES}
        for state,token in zip(expected_states,tokens): by_state[state].add(token)
        covered=all(by_state[state] for state in STATES)
        global_consistent=(covered and all(len(by_state[state])==1 for state in STATES)
                           and by_state['Locked']!=by_state['Unlocked'])
        declared={'Locked':example.token_for_locked.casefold(),
                  'Unlocked':example.token_for_unlocked.casefold()}
        mapping_adherence=all(token==declared[state]
                              for state,token in zip(expected_states,tokens))
        nonliteral=all(token not in LITERALS for token in tokens)
    answer=ANSWER_RE.search(text)
    predicted=answer.group(1).capitalize() if answer else None
    answer_valid=answer is not None
    answer_correct=predicted==example.final_answer
    expected_last_role=f'{example.final_answer}_code'
    if not matches:
        emitted_last_role='missing_or_invalid'
    else:
        last=matches[-1][1].casefold()
        if last==example.token_for_locked.casefold(): emitted_last_role='Locked_code'
        elif last==example.token_for_unlocked.casefold(): emitted_last_role='Unlocked_code'
        else: emitted_last_role='other_token'
    orientation=('Locked_token_alphabetically_first' if
                 example.token_for_locked<example.token_for_unlocked else
                 'Unlocked_token_alphabetically_first')
    return {
        'example_id':example.example_id,'sequence_length':len(example.operations),
        'mapping_orientation':orientation,'initial_state':example.initial_state,
        'final_state':example.final_answer,'expected_last_token_role':expected_last_role,
        'emitted_last_token_role':emitted_last_role,'predicted_answer':predicted,
        'answer_correct':answer_correct,'answer_valid':answer_valid,'structural':structural,
        'internal_correct':internal,'global_consistent':global_consistent,
        'mapping_adherence':mapping_adherence,'nonliteral':nonliteral,
        'last_emitted_token':matches[-1][1] if matches else None,'completion':text,
    }

@torch.inference_mode()
def evaluate(examples,batch_size=2):
    rows=[]
    for start in range(0,len(examples),batch_size):
        chunk=examples[start:start+batch_size]
        prompts=[tokenizer.apply_chat_template(prompt_messages(x),tokenize=False,
                                               add_generation_prompt=True) for x in chunk]
        batch=tokenizer(prompts,return_tensors='pt',padding=True).to(model.device)
        output=model.generate(**batch,max_new_tokens=MAX_NEW_TOKENS,do_sample=False,
            pad_token_id=tokenizer.pad_token_id,eos_token_id=tokenizer.eos_token_id)
        width=batch['input_ids'].shape[1]
        texts=tokenizer.batch_decode(output[:,width:],skip_special_tokens=True)
        rows.extend(score(example,text) for example,text in zip(chunk,texts))
        if start%20==0: print(f'Generated {min(start+len(chunk),len(examples))}/{len(examples)}')
    return rows

rows=evaluate(heldout_examples)
print('GENERATION COMPLETE:',len(rows))


In [ ]:
EXPECTED={'answer_correct':.765,'answer_valid':.985,'structural':.99,
          'internal_correct':.93,'global_consistent':.92,
          'mapping_adherence':.915,'nonliteral':.985}
actual={key:sum(bool(row[key]) for row in rows)/len(rows) for key in EXPECTED}
delta={key:actual[key]-EXPECTED[key] for key in EXPECTED}
TOLERANCE={'answer_correct':.025,'answer_valid':.02,'mapping_adherence':.015,
           'structural':.005,'internal_correct':.005,
           'global_consistent':.005,'nonliteral':.005}
print('===== REPRODUCTION CHECK =====')
print({'expected':EXPECTED,'actual':actual,'delta':delta,'tolerance':TOLERANCE})
violations={key:value for key,value in delta.items()
            if abs(value)>TOLERANCE[key]+1e-12}
if violations:
    raise RuntimeError(f'Audit generations exceed the registered reload tolerances: {violations}')

def grouped(field):
    buckets=defaultdict(list)
    for row in rows: buckets[str(row[field])].append(row)
    result=[]
    for value,items in sorted(buckets.items()):
        errors=sum(not x['answer_correct'] for x in items)
        result.append({'value':value,'n':len(items),'errors':errors,
                       'error_rate':errors/len(items)})
    return result

def association(field):
    table=grouped(field)
    counts=np.array([[x['n']-x['errors'],x['errors']] for x in table])
    chi2,p,_,_=chi2_contingency(counts)
    n=counts.sum(); v=math.sqrt(chi2/(n*max(1,min(counts.shape)-1)))
    return {'groups':table,'chi2':float(chi2),'p_value':float(p),'cramers_v':float(v),
            'max_minus_min_error_rate':max(x['error_rate'] for x in table)-min(x['error_rate'] for x in table)}

fields=('mapping_orientation','sequence_length','final_state',
        'expected_last_token_role','emitted_last_token_role')
clusters={field:association(field) for field in fields}
confusion=Counter((row['final_state'],row['predicted_answer'] or 'INVALID') for row in rows)
report={'read_only':True,'checkpoint_logical_step':150,'rows':rows,
        'reproduction':{'expected':EXPECTED,'actual':actual,'delta':delta,
                        'tolerance':TOLERANCE,'passed':True},
        'clusters':clusters,
        'answer_confusion':{f'{truth}->{prediction}':count
                            for (truth,prediction),count in sorted(confusion.items())}}
OUTPUT_JSON.write_text(json.dumps(report,indent=2,sort_keys=True)+'\n')

print('===== FINAL-ANSWER ERROR CLUSTERING =====')
for field in fields:
    print('\n',field)
    for row in clusters[field]['groups']: print(row)
    print({k:clusters[field][k] for k in ('p_value','cramers_v','max_minus_min_error_rate')})
print('\nANSWER CONFUSION:',report['answer_confusion'])
print('FULL READ-ONLY EVIDENCE SAVED:',OUTPUT_JSON)
print('STOP HERE. Dataset regeneration and training are intentionally absent.')


In [ ]:
print('===== RUN THIS LAST CELL: RECOVER CLUSTERING AFTER THE OLD TOLERANCE STOP =====')
if 'rows' not in globals() or len(rows)!=200:
    raise RuntimeError('The 200 generated rows are not in memory; run the generation cell first.')

EXPECTED={'answer_correct':.765,'answer_valid':.985,'structural':.99,
          'internal_correct':.93,'global_consistent':.92,
          'mapping_adherence':.915,'nonliteral':.985}
TOLERANCE={'answer_correct':.025,'answer_valid':.02,'mapping_adherence':.015,
           'structural':.005,'internal_correct':.005,
           'global_consistent':.005,'nonliteral':.005}
actual={key:sum(bool(row[key]) for row in rows)/len(rows) for key in EXPECTED}
delta={key:actual[key]-EXPECTED[key] for key in EXPECTED}
violations={key:value for key,value in delta.items()
            if abs(value)>TOLERANCE[key]+1e-12}
print('REPRODUCTION:',{'expected':EXPECTED,'actual':actual,'delta':delta,
                       'tolerance':TOLERANCE,'violations':violations})
if violations:
    raise RuntimeError(f'Reload evidence exceeds registered tolerances: {violations}')

def recovery_grouped(field):
    buckets=defaultdict(list)
    for row in rows: buckets[str(row[field])].append(row)
    result=[]
    for value,items in sorted(buckets.items()):
        errors=sum(not item['answer_correct'] for item in items)
        result.append({'value':value,'n':len(items),'errors':errors,
                       'error_rate':errors/len(items)})
    return result

def recovery_association(field):
    table=recovery_grouped(field)
    counts=np.array([[item['n']-item['errors'],item['errors']] for item in table])
    chi2,p,_,_=chi2_contingency(counts)
    n=counts.sum(); v=math.sqrt(chi2/(n*max(1,min(counts.shape)-1)))
    return {'groups':table,'chi2':float(chi2),'p_value':float(p),
            'cramers_v':float(v),
            'max_minus_min_error_rate':max(x['error_rate'] for x in table)-min(x['error_rate'] for x in table)}

fields=('mapping_orientation','sequence_length','final_state',
        'expected_last_token_role','emitted_last_token_role')
clusters={field:recovery_association(field) for field in fields}
confusion=Counter((row['final_state'],row['predicted_answer'] or 'INVALID') for row in rows)
report={'read_only':True,'checkpoint_logical_step':150,'rows':rows,
        'reproduction':{'expected':EXPECTED,'actual':actual,'delta':delta,
                        'tolerance':TOLERANCE,'passed':True},
        'clusters':clusters,
        'answer_confusion':{f'{truth}->{prediction}':count
                            for (truth,prediction),count in sorted(confusion.items())}}
OUTPUT_JSON.write_text(json.dumps(report,indent=2,sort_keys=True)+'\n')

print('===== FINAL-ANSWER ERROR CLUSTERING =====')
for field in fields:
    print('\n'+field)
    for item in clusters[field]['groups']: print(item)
    print({key:clusters[field][key] for key in
           ('p_value','cramers_v','max_minus_min_error_rate')})
print('\nANSWER CONFUSION:',report['answer_confusion'])
print('FULL READ-ONLY EVIDENCE SAVED:',OUTPUT_JSON)
print('DONE. Do not regenerate data or train yet.')
